# Colab Smoke Test: SD Pipeline 真实闭环验证
---

## Cell A: 环境与依赖安装（GPU）

In [ ]:
# 上传压缩包并解压
import os
from pathlib import Path
from google.colab import files

project_root = Path("/content/CEG-WM")

# 检查项目目录是否已存在
if project_root.exists():
    print(f"✓ 项目目录已存在: {project_root}")
    print(f"  跳过上传步骤\n")
else:
    print("="*80)
    print("上传 CEG-WM 项目压缩包")
    print("="*80)
    print("\n请选择 CEG-WM.zip 文件进行上传...")
    print("（提示：在本地先执行 zip -r CEG-WM.zip . 打包项目）\n")
    
    # 上传文件
    uploaded = files.upload()
    
    # 获取上传的文件名
    if not uploaded:
        raise FileNotFoundError("未上传任何文件，请重新运行此 Cell 并上传 CEG-WM.zip")
    
    uploaded_filename = list(uploaded.keys())[0]
    print(f"\n✓ 文件上传成功: {uploaded_filename}")
    
    # 解压到 /content/
    print(f"正在解压 {uploaded_filename}...")
    if uploaded_filename.endswith('.zip'):
        !unzip -q "{uploaded_filename}" -d /content/
        print(f"✓ 解压完成")
    elif uploaded_filename.endswith('.tar.gz') or uploaded_filename.endswith('.tgz'):
        !tar -xzf "{uploaded_filename}" -C /content/
        print(f"✓ 解压完成")
    else:
        raise ValueError(f"不支持的压缩格式: {uploaded_filename}，请使用 .zip 或 .tar.gz")
    
    # 验证解压后的目录
    if not project_root.exists():
        raise FileNotFoundError(
            f"解压后未找到 {project_root}。\n"
            f"请确保压缩包根目录是 CEG-WM/，而不是嵌套目录。"
        )
    
    print(f"✓ 项目目录已就绪: {project_root}\n")

# 进入项目目录并设置 PYTHONPATH
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print("="*80)
print(f"✓ 当前工作目录: {os.getcwd()}")
print(f"✓ PYTHONPATH 已设置: {sys.path[0]}")
print("="*80)

# 验证关键文件存在
print("\n验证关键文件...")
required_paths = [
    "configs/frozen_contracts.yaml",
    "configs/default.yaml",
    "configs/colab_smoke_test.yaml",
    "configs/runtime_whitelist.yaml",
    "configs/policy_path_semantics.yaml",
    "main/cli/run_embed.py",
    "main/cli/run_detect.py",
    "main/cli/run_calibrate.py",
    "main/cli/run_evaluate.py",
    "requirements.txt"
]

missing = []
for p in required_paths:
    if not (project_root / p).exists():
        missing.append(p)
        print(f"  ✗ 缺失: {p}")
    else:
        print(f"  ✓ 存在: {p}")

if missing:
    raise FileNotFoundError(f"缺失关键文件: {missing}")

print("\n✓ 所有关键文件验证通过")
print("="*80)

In [ ]:
# 安装依赖
# 注意：Colab 预装了 PyTorch，但版本可能不匹配
# 我们按项目要求安装固定版本

print("安装项目依赖（这可能需要几分钟）...")
print("="*80)

# 先升级 pip
!pip install --upgrade pip -q

# 安装关键依赖
# Colab 通常使用 torch 2.x，我们安装项目兼容的版本
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q
!pip install diffusers==0.32.0 transformers==4.45.2 accelerate==1.12.0 -q
!pip install safetensors sentencepiece protobuf -q

# 安装其他依赖
!pip install pyyaml pillow numpy scipy scikit-learn -q
!pip install matplotlib seaborn pandas tqdm -q
!pip install psutil -q

print("\n" + "="*80)
print("✓ 依赖安装完成")
print("="*80)

# 验证关键包
import torch
import diffusers
import transformers
import accelerate

print(f"\n关键包版本:")
print(f"  torch: {torch.__version__}")
print(f"  diffusers: {diffusers.__version__}")
print(f"  transformers: {transformers.__version__}")
print(f"  accelerate: {accelerate.__version__}")
print(f"  CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  CUDA 版本: {torch.version.cuda}")
    print(f"  GPU 设备: {torch.cuda.get_device_name(0)}")

In [ ]:
# Hugging Face 登录
# 用户需要提供 HF token 来下载 SD3 模型
# 不在 Notebook 中硬编码 token

from huggingface_hub import login
import getpass

print("为下载 Stable Diffusion 3 模型，需要 Hugging Face token。")
print("请访问: https://huggingface.co/settings/tokens")
print("创建或复制一个 token（需要有 read 权限）。")
print("\n注意：token 不会被显示或记录。\n")

# 交互式输入
try:
    hf_token = getpass.getpass("请输入 Hugging Face token: ")
    if hf_token:
        login(token=hf_token)
        print("\n✓ Hugging Face 登录成功")
    else:
        print("\n⚠ 未提供 token，如果模型已缓存则可继续，否则下载会失败。")
except Exception as e:
    print(f"\n⚠ 登录失败: {e}")
    print("如果模型已缓存，可以继续；否则请重新运行此 Cell 并提供有效 token。")


## Cell B: 预检（Preflight）与版本锚定

In [ ]:
# 打印版本信息
import sys
import subprocess
from pathlib import Path

print("="*80)
print("版本锚定信息")
print("="*80)

# Python 版本
print(f"Python: {sys.version}")

# Git commit
try:
    git_hash = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd="/content/CEG-WM",
        stderr=subprocess.DEVNULL
    ).decode().strip()
    print(f"Git commit: {git_hash}")
except:
    print("Git commit: <not available (zip upload)>")

# 关键依赖版本
import torch
import diffusers
import transformers

print(f"torch: {torch.__version__}")
print(f"diffusers: {diffusers.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"CUDA: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")

print("="*80)

In [ ]:
# 执行项目自检
# 跳过会下载大模型的检查项

print("执行项目自检（轻量级）...")
print("="*80)

# 检查 configs 一致性
# 注意：这里只做基本导入测试，不运行完整审计（避免耗时）
try:
    sys.path.insert(0, "/content/CEG-WM")
    
    # 验证核心模块可导入
    from main.core import config_loader
    from main.core.contracts import load_frozen_contracts
    from main.policy.runtime_whitelist import load_runtime_whitelist, load_policy_path_semantics
    
    print("✓ 核心模块导入成功")
    
    # 加载并验证事实源
    contracts = load_frozen_contracts(config_loader.FROZEN_CONTRACTS_PATH)
    whitelist = load_runtime_whitelist(config_loader.RUNTIME_WHITELIST_PATH)
    semantics = load_policy_path_semantics(config_loader.POLICY_PATH_SEMANTICS_PATH)
    
    print(f"✓ 加载 frozen_contracts: version={contracts.contract['contract_version']}")
    print(f"✓ 加载 runtime_whitelist: version={whitelist['whitelist_version']}")
    print(f"✓ 加载 policy_path_semantics: version={semantics['policy_path_semantics_version']}")
    
    # 验证一致性（简化版）
    from main.policy.runtime_whitelist import assert_consistent_with_semantics
    assert_consistent_with_semantics(whitelist, semantics)
    print("✓ Whitelist 与 semantics 一致性验证通过")
    
except Exception as e:
    print(f"✗ 自检失败: {e}")
    import traceback
    traceback.print_exc()
    raise

print("="*80)
print("✓ 预检完成")
print("="*80)

In [ ]:
# Pipeline Preflight
# 这里不实际构建 pipeline，只验证配置

print("Pipeline Preflight 检查...")
print("="*80)

try:
    from main.core import config_loader
    
    # 加载默认配置
    cfg_dict = config_loader.load_yaml_raw("configs/default.yaml")
    
    print(f"✓ 配置文件加载成功")
    print(f"  model_id: {cfg_dict.get('model_id')}")
    print(f"  device: {cfg_dict.get('device')}")
    print(f"  inference_enabled: {cfg_dict.get('inference_enabled')}")
    print(f"  watermark.lf.enabled: {cfg_dict.get('watermark', {}).get('lf', {}).get('enabled')}")
    print(f"  watermark.hf.enabled: {cfg_dict.get('watermark', {}).get('hf', {}).get('enabled')}")
    
    print("\n✓ Pipeline Preflight 通过（配置可解析）")
    print("  注意：实际 pipeline 构建将在 embed 阶段进行。")
    
except Exception as e:
    print(f"✗ Preflight 失败: {e}")
    import traceback
    traceback.print_exc()
    raise

print("="*80)

## Cell C: 准备 run_root 与最小测试输入

In [ ]:
# 创建 run_root
from pathlib import Path
import shutil

# 定义各阶段的 run_root
BASE_RUN_ROOT = Path("/content/runs")
SMOKE_RUN_ROOT = BASE_RUN_ROOT / "smoke_sd"

# 各阶段子目录
EMBED_RUN = SMOKE_RUN_ROOT / "embed"
DETECT_RUN = SMOKE_RUN_ROOT / "detect"
CALIBRATE_RUN = SMOKE_RUN_ROOT / "calibrate"
EVALUATE_RUN = SMOKE_RUN_ROOT / "evaluate"

# 清理旧运行
if SMOKE_RUN_ROOT.exists():
    print(f"清理旧运行目录: {SMOKE_RUN_ROOT}")
    shutil.rmtree(SMOKE_RUN_ROOT)

# 创建目录结构
SMOKE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(f"✓ 创建 run_root: {SMOKE_RUN_ROOT}")

# 验证
assert SMOKE_RUN_ROOT.exists() and SMOKE_RUN_ROOT.is_dir()
print(f"✓ Run_root 验证通过")

print("\n运行目录结构:")
print(f"  BASE: {BASE_RUN_ROOT}")
print(f"  SMOKE: {SMOKE_RUN_ROOT}")
print(f"    - embed: {EMBED_RUN}")
print(f"    - detect: {DETECT_RUN}")
print(f"    - calibrate: {CALIBRATE_RUN}")
print(f"    - evaluate: {EVALUATE_RUN}")

In [ ]:
# 准备最小测试输入
import json

# 测试参数
SMOKE_CONFIG = {
    "prompts": [
        "a serene landscape with mountains and a lake",
        "a futuristic city at sunset",
        "a cute cat sitting on a windowsill",
        "abstract geometric patterns in blue and gold"
    ],
    "seeds": [42, 123],
    "resolution": 512,
    "num_inference_steps": 28,
    "guidance_scale": 7.0,
    "target_fpr": 0.01,
    "num_wrong_key_samples": 128,  # 校准用 wrong-key 样本数
}

print("Smoke Test 配置:")
print("="*80)
print(json.dumps(SMOKE_CONFIG, indent=2, ensure_ascii=False))
print("="*80)

# 计算总样本数
total_embed_samples = len(SMOKE_CONFIG["prompts"]) * len(SMOKE_CONFIG["seeds"])
print(f"\n总 embed 样本数: {total_embed_samples}")
print(f"总 calibrate wrong-key 样本数: {SMOKE_CONFIG['num_wrong_key_samples']}")
print(f"\n注意：小规模 smoke test，避免 Colab 超时。")

In [ ]:
# 准备 watermark keys
# 项目应该有 key 生成器，这里使用简化版本
# 实际应使用项目的 key 生成规则

import random
import numpy as np

# 设置随机种子以保证可复现
random.seed(42)
np.random.seed(42)

# 正确的 key（embed 用）
CORRECT_KEY = "smoke_test_key_correct_42"

# Wrong keys（calibrate 用）
WRONG_KEYS = [
    f"smoke_test_key_wrong_{i}" 
    for i in range(SMOKE_CONFIG["num_wrong_key_samples"])
]

print(f"✓ 准备 watermark keys:")
print(f"  正确 key: {CORRECT_KEY}")
print(f"  错误 keys 数量: {len(WRONG_KEYS)}")
print(f"  示例错误 keys: {WRONG_KEYS[:3]}...")

# 保存 keys 到文件（供后续使用）
keys_dir = SMOKE_RUN_ROOT / "keys"
keys_dir.mkdir(exist_ok=True)

with open(keys_dir / "correct_key.txt", "w") as f:
    f.write(CORRECT_KEY)

with open(keys_dir / "wrong_keys.json", "w") as f:
    json.dump(WRONG_KEYS, f, indent=2)

print(f"✓ Keys 已保存到: {keys_dir}")

## Cell D: 真实 Embed（必须成功）

In [ ]:
# 执行 embed
import subprocess
import sys
from pathlib import Path

print("="*80)
print("执行 EMBED 流程")
print("="*80)

# 构建 CLI 命令
# 注意：必须使用 GPU，覆写 device 配置
embed_cmd = [
    sys.executable, "-m", "main.cli.run_embed",
    "--out", str(EMBED_RUN),
    "--config", "configs/default.yaml",
    "--override", "device=cuda",  # 强制使用 GPU
    "--override", f"inference_height={SMOKE_CONFIG['resolution']}",
    "--override", f"inference_width={SMOKE_CONFIG['resolution']}",
    "--override", f"inference_num_steps={SMOKE_CONFIG['num_inference_steps']}",
    "--override", f"inference_guidance_scale={SMOKE_CONFIG['guidance_scale']}",
]

print("执行命令:")
print(" ".join(embed_cmd))
print("\n" + "="*80)

# 执行
try:
    result = subprocess.run(
        embed_cmd,
        cwd="/content/CEG-WM",
        capture_output=False,  # 直接显示输出
        text=True,
        check=True
    )
    print("\n" + "="*80)
    print("✓ EMBED 执行成功")
    print("="*80)
except subprocess.CalledProcessError as e:
    print(f"\n✗ EMBED 失败: {e}")
    raise

In [ ]:
# 验证 embed 产物
import json
from pathlib import Path

print("验证 EMBED 产物...")
print("="*80)

# 检查目录结构
embed_records_dir = EMBED_RUN / "records"
embed_artifacts_dir = EMBED_RUN / "artifacts"
embed_run_closure_file = embed_records_dir / "run_closure.json"

print(f"检查目录:")
print(f"  records: {embed_records_dir.exists()}")
print(f"  artifacts: {embed_artifacts_dir.exists()}")
print(f"  run_closure: {embed_run_closure_file.exists()}")

assert embed_records_dir.exists(), "records 目录不存在"
assert embed_artifacts_dir.exists(), "artifacts 目录不存在"
assert embed_run_closure_file.exists(), "run_closure.json 不存在"

# 读取 run_closure
with open(embed_run_closure_file) as f:
    embed_closure = json.load(f)

print("\n✓ 所有必需文件存在")

# 提取关键锚点
print("\n关键锚点:")
print("="*80)

EMBED_ANCHORS = {
    "run_id": embed_closure.get("run_id"),
    "command": embed_closure.get("command"),
    "cfg_digest": embed_closure.get("cfg_digest"),
    "plan_digest": embed_closure.get("plan_digest"),
    "impl_id": embed_closure.get("impl_id"),
    "impl_version": embed_closure.get("impl_version"),
    "impl_digest": embed_closure.get("impl_identity_digest"),
    "pipeline_status": embed_closure.get("pipeline_status"),
    "status_ok": embed_closure.get("status_ok"),
    "status_reason": embed_closure.get("status_reason"),
}

for key, value in EMBED_ANCHORS.items():
    print(f"  {key}: {value}")

# 验证状态
assert EMBED_ANCHORS["command"] == "embed", "command 不是 embed"
assert EMBED_ANCHORS["status_ok"] == True, f"status_ok 不是 True: {EMBED_ANCHORS['status_reason']}"
assert EMBED_ANCHORS["pipeline_status"] in ["built", "ok"], f"pipeline_status 异常: {EMBED_ANCHORS['pipeline_status']}"

print("\n✓ EMBED 锚点验证通过")
print("="*80)

# 统计 status 分布（如果有 records）
embed_records_file = embed_records_dir / "embed_records.jsonl"
if embed_records_file.exists():
    status_counts = {"ok": 0, "absent": 0, "mismatch": 0, "fail": 0}
    with open(embed_records_file) as f:
        for line in f:
            record = json.loads(line)
            status_val = record.get("status")
            if status_val in status_counts:
                status_counts[status_val] += 1
    
    print("\nStatus 分布:")
    for status, count in status_counts.items():
        print(f"  {status}: {count}")
    
    EMBED_ANCHORS["status_counts"] = status_counts
else:
    print("\n⚠ embed_records.jsonl 不存在（可能是单样本模式）")

## Cell E: 真实 Detect（必须成功或严格失败语义）

In [ ]:
# 执行 detect
import subprocess
import sys

print("="*80)
print("执行 DETECT 流程")
print("="*80)

# 构建 CLI 命令
# 注意：detect 需要从 embed 的产物中读取输入
# 这里需要确定项目的 detect 如何接收 embed 产物
# 假设通过 --input 参数传递 embed 的 records 或 artifacts

detect_cmd = [
    sys.executable, "-m", "main.cli.run_detect",
    "--out", str(DETECT_RUN),
    "--config", "configs/default.yaml",
    "--override", "device=cuda",  # 强制使用 GPU
    # 如果项目支持，传入 embed 的产物路径
    # "--input", str(embed_records_file),  # 或者 artifacts 路径
]

print("执行命令:")
print(" ".join(detect_cmd))
print("\n" + "="*80)

# 执行
try:
    result = subprocess.run(
        detect_cmd,
        cwd="/content/CEG-WM",
        capture_output=False,
        text=True,
        check=True
    )
    print("\n" + "="*80)
    print("✓ DETECT 执行成功")
    print("="*80)
except subprocess.CalledProcessError as e:
    print(f"\n✗ DETECT 失败: {e}")
    # Detect 可能会因为依赖缺失而 fail，这是预期的失败语义
    print("\n检查是否是预期的失败语义（pipeline_missing 等）...")
    # 继续，不抛出异常

In [ ]:
# 验证 detect 产物与失败语义
import json
from pathlib import Path

print("验证 DETECT 产物...")
print("="*80)

detect_records_dir = DETECT_RUN / "records"
detect_run_closure_file = detect_records_dir / "run_closure.json"

DETECT_ANCHORS = {}
DETECT_STATUS_COUNTS = {"ok": 0, "absent": 0, "mismatch": 0, "fail": 0, "pipeline_missing": 0}

if detect_run_closure_file.exists():
    with open(detect_run_closure_file) as f:
        detect_closure = json.load(f)
    
    DETECT_ANCHORS = {
        "run_id": detect_closure.get("run_id"),
        "command": detect_closure.get("command"),
        "cfg_digest": detect_closure.get("cfg_digest"),
        "impl_id": detect_closure.get("impl_id"),
        "impl_version": detect_closure.get("impl_version"),
        "impl_digest": detect_closure.get("impl_identity_digest"),
        "pipeline_status": detect_closure.get("pipeline_status"),
        "status_ok": detect_closure.get("status_ok"),
        "status_reason": detect_closure.get("status_reason"),
    }
    
    print("关键锚点:")
    for key, value in DETECT_ANCHORS.items():
        print(f"  {key}: {value}")
    
    # 统计 status 分布
    detect_records_file = detect_records_dir / "detect_records.jsonl"
    if detect_records_file.exists():
        with open(detect_records_file) as f:
            for line in f:
                record = json.loads(line)
                status_val = record.get("status")
                if status_val in DETECT_STATUS_COUNTS:
                    DETECT_STATUS_COUNTS[status_val] += 1
                
                # 检查 pipeline_missing
                failure_reason = record.get("failure_reason", "")
                if "pipeline_missing" in str(failure_reason).lower():
                    DETECT_STATUS_COUNTS["pipeline_missing"] += 1
        
        print("\nStatus 分布:")
        for status, count in DETECT_STATUS_COUNTS.items():
            print(f"  {status}: {count}")
    
    # 验证：在正常条件下 pipeline_missing 应为 0
    if DETECT_STATUS_COUNTS["pipeline_missing"] > 0:
        print(f"\n⚠ 警告: pipeline_missing 计数 = {DETECT_STATUS_COUNTS['pipeline_missing']}")
        print("  这表明 detect 缺失 pipeline 依赖。")
        print("  在 smoke test 中应为 0（因为我们显式构建了 pipeline）。")
    else:
        print("\n✓ pipeline_missing 计数 = 0（符合预期）")
    
    # 验证：status!=ok 时不存在 content_score
    with open(detect_records_file) as f:
        for line in f:
            record = json.loads(line)
            status_val = record.get("status")
            content_score = record.get("content_score")
            
            if status_val != "ok" and content_score is not None:
                print(f"\n✗ 错误: status={status_val} 但 content_score 存在: {content_score}")
                raise AssertionError("违反失败语义: status!=ok 时不应有 content_score")
    
    print("\n✓ 失败语义一致性验证通过（status!=ok 时无 content_score）")
    
else:
    print("⚠ detect run_closure.json 不存在")
    print("  Detect 可能完全失败，检查日志。")

print("="*80)

## Cell F: 最小校准 Calibrate（Wrong-Key Null，小规模）

In [ ]:
# 执行 calibrate
import subprocess
import sys

print("="*80)
print("执行 CALIBRATE 流程")
print("="*80)

# 构建 CLI 命令
calibrate_cmd = [
    sys.executable, "-m", "main.cli.run_calibrate",
    "--out", str(CALIBRATE_RUN),
    "--config", "configs/default.yaml",
    "--override", "device=cuda",
    "--override", f"target_fpr={SMOKE_CONFIG['target_fpr']}",
    # 如果项目支持指定 wrong-key 样本数
    # "--override", f"calibrate.num_samples={SMOKE_CONFIG['num_wrong_key_samples']}",
]

print("执行命令:")
print(" ".join(calibrate_cmd))
print("\n" + "="*80)

# 执行
try:
    result = subprocess.run(
        calibrate_cmd,
        cwd="/content/CEG-WM",
        capture_output=False,
        text=True,
        check=True
    )
    print("\n" + "="*80)
    print("✓ CALIBRATE 执行成功")
    print("="*80)
except subprocess.CalledProcessError as e:
    print(f"\n✗ CALIBRATE 失败: {e}")
    raise

In [ ]:
# 验证 calibrate 产物
import json
from pathlib import Path

print("验证 CALIBRATE 产物...")
print("="*80)

calibrate_records_dir = CALIBRATE_RUN / "records"
calibrate_artifacts_dir = CALIBRATE_RUN / "artifacts"
calibrate_run_closure_file = calibrate_records_dir / "run_closure.json"

assert calibrate_run_closure_file.exists(), "calibrate run_closure.json 不存在"

with open(calibrate_run_closure_file) as f:
    calibrate_closure = json.load(f)

CALIBRATE_ANCHORS = {
    "run_id": calibrate_closure.get("run_id"),
    "command": calibrate_closure.get("command"),
    "cfg_digest": calibrate_closure.get("cfg_digest"),
    "target_fpr": calibrate_closure.get("target_fpr"),
    "thresholds_digest": calibrate_closure.get("thresholds_digest"),
    "threshold_metadata_digest": calibrate_closure.get("threshold_metadata_digest"),
    "status_ok": calibrate_closure.get("status_ok"),
    "status_reason": calibrate_closure.get("status_reason"),
}

print("关键锚点:")
for key, value in CALIBRATE_ANCHORS.items():
    print(f"  {key}: {value}")

# 验证状态
assert CALIBRATE_ANCHORS["command"] == "calibrate", "command 不是 calibrate"
assert CALIBRATE_ANCHORS["status_ok"] == True, f"status_ok 不是 True: {CALIBRATE_ANCHORS['status_reason']}"

# 检查 thresholds 工件
thresholds_file = calibrate_artifacts_dir / "thresholds.json"
threshold_metadata_file = calibrate_artifacts_dir / "threshold_metadata.json"

assert thresholds_file.exists(), "thresholds.json 不存在"
assert threshold_metadata_file.exists(), "threshold_metadata.json 不存在"

with open(thresholds_file) as f:
    thresholds = json.load(f)

with open(threshold_metadata_file) as f:
    threshold_metadata = json.load(f)

print("\nThresholds 工件:")
print(f"  thresholds.json: {thresholds_file}")
print(f"  threshold_metadata.json: {threshold_metadata_file}")
print(f"\n  样本数: {threshold_metadata.get('num_samples', 'N/A')}")
print(f"  target_fpr: {threshold_metadata.get('target_fpr', 'N/A')}")

# 如果有阈值摘要，打印出来
if isinstance(thresholds, dict):
    print(f"\n  阈值摘要:")
    for key, value in list(thresholds.items())[:5]:  # 只显示前 5 个
        print(f"    {key}: {value}")

print("\n✓ CALIBRATE 锚点验证通过")
print("="*80)

## Cell G: 最小评测 Evaluate（只读 Thresholds）

In [ ]:
# 执行 evaluate
import subprocess
import sys

print("="*80)
print("执行 EVALUATE 流程")
print("="*80)

# 构建 CLI 命令
# 注意：evaluate 必须明确传入 thresholds 工件路径（只读）
evaluate_cmd = [
    sys.executable, "-m", "main.cli.run_evaluate",
    "--out", str(EVALUATE_RUN),
    "--config", "configs/default.yaml",
    "--override", "device=cuda",
    # 如果项目支持指定 thresholds 路径
    # "--override", f"thresholds_path={thresholds_file}",
]

print("执行命令:")
print(" ".join(evaluate_cmd))
print("\n" + "="*80)

# 执行
try:
    result = subprocess.run(
        evaluate_cmd,
        cwd="/content/CEG-WM",
        capture_output=False,
        text=True,
        check=True
    )
    print("\n" + "="*80)
    print("✓ EVALUATE 执行成功")
    print("="*80)
except subprocess.CalledProcessError as e:
    print(f"\n✗ EVALUATE 失败: {e}")
    raise

In [ ]:
# 验证 evaluate 产物
import json
from pathlib import Path

print("验证 EVALUATE 产物...")
print("="*80)

evaluate_records_dir = EVALUATE_RUN / "records"
evaluate_run_closure_file = evaluate_records_dir / "run_closure.json"

EVALUATE_ANCHORS = {}

if evaluate_run_closure_file.exists():
    with open(evaluate_run_closure_file) as f:
        evaluate_closure = json.load(f)
    
    EVALUATE_ANCHORS = {
        "run_id": evaluate_closure.get("run_id"),
        "command": evaluate_closure.get("command"),
        "cfg_digest": evaluate_closure.get("cfg_digest"),
        "thresholds_digest": evaluate_closure.get("thresholds_digest"),
        "threshold_metadata_digest": evaluate_closure.get("threshold_metadata_digest"),
        "status_ok": evaluate_closure.get("status_ok"),
        "status_reason": evaluate_closure.get("status_reason"),
    }
    
    print("关键锚点:")
    for key, value in EVALUATE_ANCHORS.items():
        print(f"  {key}: {value}")
    
    # 查找评测结果
    evaluate_results_file = evaluate_records_dir / "evaluation_results.json"
    if evaluate_results_file.exists():
        with open(evaluate_results_file) as f:
            eval_results = json.load(f)
        
        print("\n评测结果:")
        print(f"  TPR@FPR: {eval_results.get('tpr_at_fpr', 'N/A')}")
        print(f"  Reject Rate: {eval_results.get('reject_rate', 'N/A')}")
        
        EVALUATE_ANCHORS["tpr_at_fpr"] = eval_results.get("tpr_at_fpr")
        EVALUATE_ANCHORS["reject_rate"] = eval_results.get("reject_rate")
    else:
        print("\n⚠ evaluation_results.json 不存在")
    
    print("\n✓ EVALUATE 锚点验证通过")
else:
    print("⚠ evaluate run_closure.json 不存在")

print("="*80)

## Cell H: 闭环一致性断言（必须）

In [ ]:
# 闭环一致性断言
print("="*80)
print("闭环一致性断言")
print("="*80)

# 断言 1: cfg_digest 一致性
print("\n1. cfg_digest 一致性检查:")
cfg_digests = {
    "embed": EMBED_ANCHORS.get("cfg_digest"),
    "detect": DETECT_ANCHORS.get("cfg_digest"),
    "calibrate": CALIBRATE_ANCHORS.get("cfg_digest"),
    "evaluate": EVALUATE_ANCHORS.get("cfg_digest"),
}

print(f"  Embed cfg_digest: {cfg_digests['embed']}")
print(f"  Detect cfg_digest: {cfg_digests['detect']}")
print(f"  Calibrate cfg_digest: {cfg_digests['calibrate']}")
print(f"  Evaluate cfg_digest: {cfg_digests['evaluate']}")

# 注意：cfg_digest 可能因为不同阶段的 override 而不同
# 这里只检查非空
for stage, digest in cfg_digests.items():
    assert digest and digest != "unknown", f"{stage} cfg_digest 缺失"

print("  ✓ 所有阶段都有 cfg_digest")

# 断言 2: thresholds_digest 一致性（evaluate 应与 calibrate 一致）
print("\n2. thresholds_digest 一致性检查:")
calibrate_thresholds_digest = CALIBRATE_ANCHORS.get("thresholds_digest")
evaluate_thresholds_digest = EVALUATE_ANCHORS.get("thresholds_digest")

print(f"  Calibrate thresholds_digest: {calibrate_thresholds_digest}")
print(f"  Evaluate thresholds_digest: {evaluate_thresholds_digest}")

if calibrate_thresholds_digest and evaluate_thresholds_digest:
    assert calibrate_thresholds_digest == evaluate_thresholds_digest, \
        "thresholds_digest 不一致（evaluate 必须使用 calibrate 的阈值）"
    print("  ✓ thresholds_digest 一致")
else:
    print("  ⚠ 某个阶段的 thresholds_digest 缺失，跳过一致性检查")

# 断言 3: status!=ok 时不存在 content_score（已在 E-2 检查）
print("\n3. 失败语义一致性:")
print("  ✓ 已在 Detect 验证阶段检查（status!=ok 时无 content_score）")

# 断言 4: pipeline_missing 计数为 0
print("\n4. pipeline_missing 计数检查:")
pipeline_missing_count = DETECT_STATUS_COUNTS.get("pipeline_missing", 0)
print(f"  pipeline_missing 计数: {pipeline_missing_count}")

if pipeline_missing_count == 0:
    print("  ✓ pipeline_missing = 0（符合预期）")
else:
    print(f"  ⚠ 警告: pipeline_missing = {pipeline_missing_count}（预期为 0）")
    # 这里可以选择是否断言失败
    # assert pipeline_missing_count == 0, "pipeline_missing 不为 0"

# 断言 5: 输出目录未发生目录逃逸（简化检查）
print("\n5. 输出目录路径合规性检查:")
all_runs = [EMBED_RUN, DETECT_RUN, CALIBRATE_RUN, EVALUATE_RUN]
for run_path in all_runs:
    # 检查是否在 run_root 下
    try:
        run_path.relative_to(SMOKE_RUN_ROOT)
        print(f"  ✓ {run_path.name} 在 run_root 下")
    except ValueError:
        raise AssertionError(f"目录逃逸: {run_path} 不在 {SMOKE_RUN_ROOT} 下")

print("\n" + "="*80)
print("✓ 所有闭环一致性断言通过")
print("="*80)

## Cell I: 可选 - 运行 pytest

In [ ]:
# 运行 pytest（可选）
# 注意：这可能会很耗时，仅在需要时运行

print("="*80)
print("运行 pytest（可选）")
print("="*80)
print("\n⚠ 此步骤可选，可能耗时较长。")
print("如需跳过，请注释掉下面的代码。\n")

# 取消下面的注释以运行 pytest
# import subprocess
# import sys
# 
# try:
#     result = subprocess.run(
#         [sys.executable, "-m", "pytest", "-q"],
#         cwd="/content/CEG-WM",
#         capture_output=True,
#         text=True,
#         timeout=300  # 5 分钟超时
#     )
#     print(result.stdout)
#     if result.stderr:
#         print("STDERR:")
#         print(result.stderr)
#     
#     if result.returncode == 0:
#         print("\n✓ pytest 通过")
#     else:
#         print(f"\n⚠ pytest 返回码: {result.returncode}")
# except subprocess.TimeoutExpired:
#     print("\n⚠ pytest 超时（5 分钟）")
# except Exception as e:
#     print(f"\n⚠ pytest 执行失败: {e}")

print("跳过 pytest（已注释）")

## Cell J: Smoke Test 总结

In [ ]:
# 生成 Smoke Test 总结表
import json
from datetime import datetime

print("="*80)
print("SMOKE TEST 总结")
print("="*80)

summary = {
    "run_root": str(SMOKE_RUN_ROOT),
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "stages": {
        "embed": {
            "status": "PASS" if EMBED_ANCHORS.get("status_ok") else "FAIL",
            "run_id": EMBED_ANCHORS.get("run_id"),
            "cfg_digest": EMBED_ANCHORS.get("cfg_digest"),
            "plan_digest": EMBED_ANCHORS.get("plan_digest"),
            "impl_id": EMBED_ANCHORS.get("impl_id"),
            "impl_version": EMBED_ANCHORS.get("impl_version"),
            "impl_digest": EMBED_ANCHORS.get("impl_digest"),
            "pipeline_status": EMBED_ANCHORS.get("pipeline_status"),
        },
        "detect": {
            "status": "PASS" if DETECT_ANCHORS.get("status_ok") else "FAIL",
            "run_id": DETECT_ANCHORS.get("run_id"),
            "cfg_digest": DETECT_ANCHORS.get("cfg_digest"),
            "pipeline_status": DETECT_ANCHORS.get("pipeline_status"),
        },
        "calibrate": {
            "status": "PASS" if CALIBRATE_ANCHORS.get("status_ok") else "FAIL",
            "run_id": CALIBRATE_ANCHORS.get("run_id"),
            "cfg_digest": CALIBRATE_ANCHORS.get("cfg_digest"),
            "thresholds_digest": CALIBRATE_ANCHORS.get("thresholds_digest"),
            "threshold_metadata_digest": CALIBRATE_ANCHORS.get("threshold_metadata_digest"),
            "target_fpr": CALIBRATE_ANCHORS.get("target_fpr"),
        },
        "evaluate": {
            "status": "PASS" if EVALUATE_ANCHORS.get("status_ok") else "FAIL",
            "run_id": EVALUATE_ANCHORS.get("run_id"),
            "cfg_digest": EVALUATE_ANCHORS.get("cfg_digest"),
            "thresholds_digest": EVALUATE_ANCHORS.get("thresholds_digest"),
            "tpr_at_fpr": EVALUATE_ANCHORS.get("tpr_at_fpr"),
            "reject_rate": EVALUATE_ANCHORS.get("reject_rate"),
        },
    },
    "status_counts": {
        "embed": EMBED_ANCHORS.get("status_counts", {}),
        "detect": DETECT_STATUS_COUNTS,
    },
    "consistency_checks": {
        "cfg_digest_present": all(cfg_digests.values()),
        "thresholds_digest_consistent": (
            CALIBRATE_ANCHORS.get("thresholds_digest") == EVALUATE_ANCHORS.get("thresholds_digest")
            if CALIBRATE_ANCHORS.get("thresholds_digest") and EVALUATE_ANCHORS.get("thresholds_digest")
            else None
        ),
        "pipeline_missing_count": DETECT_STATUS_COUNTS.get("pipeline_missing", 0),
        "path_policy_compliant": True,  # 已在 H-1 检查
    },
}

# 打印总结
print(json.dumps(summary, indent=2, ensure_ascii=False))

# 保存总结到文件
summary_file = SMOKE_RUN_ROOT / "smoke_test_summary.json"
with open(summary_file, "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n" + "="*80)
print(f"✓ 总结已保存到: {summary_file}")
print("="*80)

# 最终判定
all_passed = all([
    summary["stages"]["embed"]["status"] == "PASS",
    summary["stages"]["detect"]["status"] == "PASS",
    summary["stages"]["calibrate"]["status"] == "PASS",
    summary["stages"]["evaluate"]["status"] == "PASS",
    summary["consistency_checks"]["pipeline_missing_count"] == 0,
])

print("\n" + "="*80)
if all_passed:
    print("🎉 SMOKE TEST 全部通过！")
else:
    print("⚠ SMOKE TEST 部分失败，请检查上述输出。")
print("="*80)

In [ ]:
# 下载结果到本地
# 如果需要将结果下载到本地，取消下面的注释

from google.colab import files

print("下载 smoke test 总结...")
files.download(str(summary_file))
# 
# 也可以下载整个 run_root（需要先打包）
import shutil
import tarfile
# 
archive_path = "/content/smoke_test_results.tar.gz"
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(SMOKE_RUN_ROOT, arcname="smoke_sd")

print(f"\n下载完整结果: {archive_path}")
files.download(archive_path)